# TOI-1130 System

In [ ]:
# TOI-1130 system animation
# 4:3 GIF — outer orbit fully fitted into rectangular frame

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# -----------------------------
# Orbital parameters
# -----------------------------

a_b = 0.046
a_c = 0.074

P_b = 4.07
P_c = 8.35

# Visual orbit scaling:
# outer orbit is fitted into 4:3 rectangular frame
outer_rx = 0.88
outer_ry = 0.52

inner_scale = a_b / a_c
inner_rx = outer_rx * inner_scale
inner_ry = outer_ry * inner_scale

# Visual sizes
star_size = 1200
planet_b_size = 170
planet_c_size = 320

# -----------------------------
# Animation timing
# -----------------------------

fps = 24
duration_seconds = 12
frames = fps * duration_seconds

duration_days = P_c * 2

t = np.linspace(0, duration_days, frames)

theta_b = 2 * np.pi * t / P_b
theta_c = 2 * np.pi * t / P_c

x_b = inner_rx * np.cos(theta_b)
y_b = inner_ry * np.sin(theta_b)

x_c = outer_rx * np.cos(theta_c)
y_c = outer_ry * np.sin(theta_c)

# -----------------------------
# Figure setup: true 4:3
# -----------------------------

fig = plt.figure(figsize=(12, 9), dpi=120, facecolor="black")
ax = fig.add_axes([0, 0, 1, 1])

ax.set_facecolor("black")
ax.axis("off")

# Fixed rectangular coordinate field
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(-0.75, 0.75)

# IMPORTANT:
# no equal aspect here; we deliberately use the full 4:3 rectangular field
ax.set_aspect("auto")

# -----------------------------
# Background stars
# -----------------------------

rng = np.random.default_rng(42)

stars_x = rng.uniform(-1.0, 1.0, 650)
stars_y = rng.uniform(-0.75, 0.75, 650)
stars_s = rng.uniform(0.2, 2.4, 650)

ax.scatter(
    stars_x,
    stars_y,
    s=stars_s,
    color="white",
    alpha=0.45,
    zorder=0
)

# -----------------------------
# Orbits
# -----------------------------

theta = np.linspace(0, 2 * np.pi, 1200)

ax.plot(
    inner_rx * np.cos(theta),
    inner_ry * np.sin(theta),
    color="#4ea3ff",
    lw=1.5,
    alpha=0.45,
    zorder=1
)

ax.plot(
    outer_rx * np.cos(theta),
    outer_ry * np.sin(theta),
    color="#ff9d3c",
    lw=1.5,
    alpha=0.45,
    zorder=1
)

# -----------------------------
# Star glow
# -----------------------------

ax.scatter([0], [0], s=9000, color="#ff6a00", alpha=0.055, zorder=2)
ax.scatter([0], [0], s=5200, color="#ffae00", alpha=0.12, zorder=3)
ax.scatter([0], [0], s=2600, color="#ffd36a", alpha=0.20, zorder=4)

star = ax.scatter(
    [0], [0],
    s=star_size,
    color="#ffe08a",
    edgecolor="#fff5cf",
    linewidth=1.5,
    zorder=5
)

# -----------------------------
# Planets
# -----------------------------

planet_b = ax.scatter(
    [], [],
    s=planet_b_size,
    color="#2f74c0",
    edgecolor="#bfe2ff",
    linewidth=1.0,
    zorder=8
)

planet_c = ax.scatter(
    [], [],
    s=planet_c_size,
    color="#c78647",
    edgecolor="#ffd0a0",
    linewidth=1.0,
    zorder=8
)

# -----------------------------
# Motion trails
# -----------------------------

trail_b, = ax.plot([], [], lw=2.2, color="#4ea3ff", alpha=0.6, zorder=7)
trail_c, = ax.plot([], [], lw=2.2, color="#ff9d3c", alpha=0.6, zorder=7)

# -----------------------------
# Titles / labels
# -----------------------------

ax.text(
    0,
    0.66,
    "TOI-1130 SYSTEM",
    color="white",
    fontsize=24,
    ha="center",
    va="center"
)

ax.text(
    0,
    0.59,
    "2:1 orbital resonance",
    color="white",
    fontsize=12,
    alpha=0.7,
    ha="center",
    va="center"
)

label_b = ax.text(
    0, 0,
    "TOI-1130 b",
    color="#9fd0ff",
    fontsize=10,
    ha="left",
    va="bottom"
)

label_c = ax.text(
    0, 0,
    "TOI-1130 c",
    color="#ffd0a0",
    fontsize=10,
    ha="left",
    va="bottom"
)

# -----------------------------
# Update
# -----------------------------

def update(i):
    xb, yb = x_b[i], y_b[i]
    xc, yc = x_c[i], y_c[i]

    planet_b.set_offsets([[xb, yb]])
    planet_c.set_offsets([[xc, yc]])

    trail_len = 55
    start = max(0, i - trail_len)

    trail_b.set_data(x_b[start:i + 1], y_b[start:i + 1])
    trail_c.set_data(x_c[start:i + 1], y_c[start:i + 1])

    label_b.set_position((xb + 0.035, yb + 0.025))
    label_c.set_position((xc + 0.035, yc + 0.025))

    return planet_b, planet_c, trail_b, trail_c, label_b, label_c

# -----------------------------
# Animation
# -----------------------------

anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=1000 / fps,
    blit=True
)

# -----------------------------
# Save GIF
# -----------------------------

writer = PillowWriter(fps=fps)

anim.save(
    "animations/toi1130_system_fitted_4x3.gif",
    writer=writer,
    savefig_kwargs={
        "facecolor": "black",
        "pad_inches": 0
    }
)

print("Saved: toi1130_system_fitted_4x3.gif")

plt.close(fig)

# Photometry

In [ ]:
# TOI-1130 transit photometry animation (DARK THEME)
# Output:
# animations/toi1130_transit_photometry_dark.gif

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ---------------------------------------------------
# Output directory
# ---------------------------------------------------

OUTPUT_DIR = Path("animations")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_GIF = OUTPUT_DIR / "toi1130_transit_photometry_dark.gif"

# ---------------------------------------------------
# Synthetic transit model
# ---------------------------------------------------

rng = np.random.default_rng(42)

t = np.linspace(-3.0, 3.0, 1400)

baseline = 1.00045
depth = 0.00245

t1 = -0.90
t2 = -0.72
t3 = 0.72
t4 = 0.90

def smoothstep(x):
    x = np.clip(x, 0, 1)
    return x * x * (3 - 2 * x)

def transit_model(time):

    flux = np.full_like(time, baseline)

    ingress = (time >= t1) & (time < t2)
    full     = (time >= t2) & (time <= t3)
    egress   = (time > t3) & (time <= t4)

    flux[ingress] = (
        baseline
        - depth * smoothstep((time[ingress] - t1) / (t2 - t1))
    )

    flux[full] = baseline - depth

    flux[egress] = (
        baseline
        - depth * (
            1 - smoothstep((time[egress] - t3) / (t4 - t3))
        )
    )

    # slight curvature inside transit
    inside = (time >= t2) & (time <= t3)

    flux[inside] += 0.00018 * ((time[inside] / t3) ** 2)

    # tiny baseline slope
    flux += 0.00002 * time

    return flux

model = transit_model(t)

# ---------------------------------------------------
# Noisy photometry
# ---------------------------------------------------

noise_sigma = 0.00028

raw_flux = (
    model
    + rng.normal(0, noise_sigma, size=t.size)
)

# ---------------------------------------------------
# Binning
# ---------------------------------------------------

n_bins = 34

bin_edges = np.linspace(t.min(), t.max(), n_bins + 1)

bin_centers = 0.5 * (
    bin_edges[:-1]
    + bin_edges[1:]
)

bin_flux = np.zeros(n_bins)
bin_resid = np.zeros(n_bins)

for i in range(n_bins):

    mask = (
        (t >= bin_edges[i])
        & (t < bin_edges[i + 1])
    )

    bin_flux[i] = np.mean(raw_flux[mask])

    bin_resid[i] = np.mean(
        (raw_flux[mask] - model[mask]) / noise_sigma
    )

residual_raw = (
    raw_flux - model
) / noise_sigma

# ---------------------------------------------------
# Figure setup
# ---------------------------------------------------

BG = "#05070b"
FG = "#e8edf5"
GRID = "#3c4658"

fig = plt.figure(
    figsize=(12, 9),
    dpi=120,
    facecolor=BG
)

gs = fig.add_gridspec(
    2,
    1,
    height_ratios=[2.2, 1.0],
    hspace=0.03
)

ax_flux = fig.add_subplot(gs[0])
ax_res  = fig.add_subplot(gs[1], sharex=ax_flux)

for ax in (ax_flux, ax_res):

    ax.set_facecolor(BG)

    ax.tick_params(
        direction="in",
        top=True,
        right=True,
        colors=FG,
        labelsize=14
    )

    for spine in ax.spines.values():
        spine.set_color(FG)
        spine.set_linewidth(1.2)

    ax.grid(
        color=GRID,
        alpha=0.15,
        linewidth=0.8
    )

# ---------------------------------------------------
# Limits / labels
# ---------------------------------------------------

ax_flux.set_xlim(-3.05, 3.05)
ax_flux.set_ylim(0.99725, 1.00115)

ax_res.set_ylim(-7.5, 7.5)

ax_flux.set_ylabel(
    "Normalized flux",
    fontsize=18,
    color=FG
)

ax_res.set_ylabel(
    "Residual [σ]",
    fontsize=16,
    color=FG
)

ax_res.set_xlabel(
    "Time from mid-transit [hours]",
    fontsize=16,
    color=FG
)

# ---------------------------------------------------
# Title
# ---------------------------------------------------

ax_flux.text(
    2.55,
    0.99752,
    "NIRISS Order 1",
    fontsize=20,
    ha="right",
    va="bottom",
    color=FG
)

# ---------------------------------------------------
# Static model
# ---------------------------------------------------

ax_flux.plot(
    t,
    model,
    color="#ff5b57",
    lw=2.0,
    zorder=4
)

ax_res.axhline(
    0,
    color="#ff5b57",
    lw=1.8,
    zorder=4
)

# ---------------------------------------------------
# Animated layers
# ---------------------------------------------------

raw_scatter_flux = ax_flux.scatter(
    [],
    [],
    s=8,
    color="#a8b2c3",
    alpha=0.20,
    edgecolors="none",
    zorder=1
)

bin_scatter_flux = ax_flux.scatter(
    [],
    [],
    s=58,
    color="#3b82ff",
    edgecolors="#dfe9ff",
    linewidths=0.7,
    zorder=6
)

raw_scatter_res = ax_res.scatter(
    [],
    [],
    s=8,
    color="#a8b2c3",
    alpha=0.15,
    edgecolors="none",
    zorder=1
)

bin_scatter_res = ax_res.scatter(
    [],
    [],
    s=58,
    color="#3b82ff",
    edgecolors="#dfe9ff",
    linewidths=0.7,
    zorder=6
)

# ---------------------------------------------------
# Time markers
# ---------------------------------------------------

time_marker_flux = ax_flux.axvline(
    t.min(),
    color=FG,
    lw=1.0,
    alpha=0.25
)

time_marker_res = ax_res.axvline(
    t.min(),
    color=FG,
    lw=1.0,
    alpha=0.25
)

time_label = ax_flux.text(
    -2.95,
    1.00095,
    "",
    fontsize=15,
    ha="left",
    va="top",
    color=FG
)

# ---------------------------------------------------
# Animation settings
# ---------------------------------------------------

fps = 24
duration_seconds = 10

frames = fps * duration_seconds

# ---------------------------------------------------
# Update
# ---------------------------------------------------

def update(frame):

    progress = frame / (frames - 1)

    current_time = (
        t.min()
        + progress * (t.max() - t.min())
    )

    raw_mask = t <= current_time
    bin_mask = bin_centers <= current_time

    raw_scatter_flux.set_offsets(
        np.column_stack([
            t[raw_mask],
            raw_flux[raw_mask]
        ])
    )

    raw_scatter_res.set_offsets(
        np.column_stack([
            t[raw_mask],
            residual_raw[raw_mask]
        ])
    )

    bin_scatter_flux.set_offsets(
        np.column_stack([
            bin_centers[bin_mask],
            bin_flux[bin_mask]
        ])
    )

    bin_scatter_res.set_offsets(
        np.column_stack([
            bin_centers[bin_mask],
            bin_resid[bin_mask]
        ])
    )

    time_marker_flux.set_xdata([
        current_time,
        current_time
    ])

    time_marker_res.set_xdata([
        current_time,
        current_time
    ])

    time_label.set_text(
        f"t = {current_time:+.2f} h"
    )

    return (
        raw_scatter_flux,
        raw_scatter_res,
        bin_scatter_flux,
        bin_scatter_res,
        time_marker_flux,
        time_marker_res,
        time_label
    )

# ---------------------------------------------------
# Animation
# ---------------------------------------------------

anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=1000 / fps,
    blit=True
)

# ---------------------------------------------------
# Save GIF
# ---------------------------------------------------

writer = PillowWriter(fps=fps)

anim.save(
    OUTPUT_GIF,
    writer=writer,
    savefig_kwargs={
        "facecolor": BG,
        "pad_inches": 0.05
    }
)

print(f"Saved: {OUTPUT_GIF}")

plt.close(fig)

# Spectra

In [ ]:
# TOI-1130 b animated transmission spectrum
# Dark theme GIF
# Output:
# animations/toi1130_transmission_spectrum_dark.gif

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ---------------------------------------------------
# Output directory
# ---------------------------------------------------

OUTPUT_DIR = Path("animations")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_GIF = OUTPUT_DIR / "toi1130_transmission_spectrum_dark.gif"

# ---------------------------------------------------
# Synthetic spectrum model
# ---------------------------------------------------

rng = np.random.default_rng(1130)

w = np.linspace(0.5, 5.2, 900)

def gaussian(x, mu, sigma, amp):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def best_fit_model(x):
    continuum = 2310 + 55 * np.exp(-(x - 0.5) / 0.35)
    continuum += 8 * np.sin(2.7 * x) + 5 * np.sin(8.0 * x)

    features = (
        gaussian(x, 1.38, 0.055, 42) +
        gaussian(x, 1.78, 0.070, 35) +
        gaussian(x, 2.35, 0.080, 28) +
        gaussian(x, 2.72, 0.090, 55) +
        gaussian(x, 3.35, 0.070, 25) +
        gaussian(x, 4.02, 0.045, 62) +
        gaussian(x, 4.32, 0.100, 95)
    )

    return continuum + features

model = best_fit_model(w)

model_no_water = model - (
    gaussian(w, 1.38, 0.070, 45) +
    gaussian(w, 1.78, 0.090, 38) +
    gaussian(w, 2.35, 0.100, 32) +
    gaussian(w, 2.72, 0.120, 42)
)

model_no_methane = model - gaussian(w, 3.35, 0.095, 35)
model_no_co2 = model - gaussian(w, 4.32, 0.130, 92)
model_no_so2 = model - gaussian(w, 4.02, 0.065, 60)

# ---------------------------------------------------
# Synthetic observed points
# ---------------------------------------------------

niriss_w = np.linspace(0.65, 2.7, 42)
nirspec_w = np.linspace(2.8, 5.15, 48)
phot_w = np.array([0.65, 0.78])

niriss_y = best_fit_model(niriss_w) + rng.normal(0, 38, size=niriss_w.size)
nirspec_y = best_fit_model(nirspec_w) + rng.normal(0, 45, size=nirspec_w.size)
phot_y = best_fit_model(phot_w) + rng.normal(0, 35, size=phot_w.size)

niriss_err = rng.uniform(25, 70, size=niriss_w.size)
nirspec_err = rng.uniform(30, 85, size=nirspec_w.size)
phot_err = rng.uniform(45, 95, size=phot_w.size)

# Residuals against best-fit model
all_w = np.concatenate([phot_w, niriss_w, nirspec_w])
all_y = np.concatenate([phot_y, niriss_y, nirspec_y])
all_err = np.concatenate([phot_err, niriss_err, nirspec_err])

resid = (all_y - best_fit_model(all_w)) / all_err

sort_idx = np.argsort(all_w)
all_w = all_w[sort_idx]
resid = resid[sort_idx]

# ---------------------------------------------------
# Figure setup
# ---------------------------------------------------

BG = "#05070b"
FG = "#e8edf5"
GRID = "#3c4658"

fig = plt.figure(figsize=(14, 9), dpi=120, facecolor=BG)

gs = fig.add_gridspec(
    2,
    1,
    height_ratios=[3.5, 1.1],
    hspace=0.04
)

ax = fig.add_subplot(gs[0])
axr = fig.add_subplot(gs[1], sharex=ax)

for a in (ax, axr):
    a.set_facecolor(BG)
    a.tick_params(
        direction="in",
        top=True,
        right=True,
        colors=FG,
        labelsize=14
    )

    for spine in a.spines.values():
        spine.set_color(FG)
        spine.set_linewidth(1.2)

    a.grid(
        color=GRID,
        alpha=0.14,
        linewidth=0.8
    )

ax.set_xlim(0.5, 5.2)
ax.set_ylim(2215, 2460)
axr.set_ylim(-2.6, 2.6)

ax.set_ylabel("Transit depth [ppm]", fontsize=20, color=FG)
axr.set_ylabel("Residual [σ]", fontsize=15, color=FG)
axr.set_xlabel("Wavelength [micron]", fontsize=22, color=FG)

# ---------------------------------------------------
# Molecular bands
# ---------------------------------------------------

bands = [
    ("H$_2$O", 1.30, 1.55, "#3b82ff"),
    ("H$_2$O", 1.75, 1.95, "#3b82ff"),
    ("H$_2$O", 2.20, 2.65, "#3b82ff"),
    ("CH$_4$", 3.20, 3.60, "#ff5b57"),
    ("SO$_2$", 3.92, 4.12, "#ff5b57"),
    ("CO$_2$", 4.18, 4.62, "#ffd166"),
]

band_patches = []

for label, x0, x1, color in bands:
    patch = ax.axvspan(
        x0,
        x1,
        color=color,
        alpha=0.0,
        zorder=0
    )
    band_patches.append((patch, label, x0, x1, color))

mol_labels = [
    ax.text(1.43, 2408, "H$_2$O", color="#4ea3ff", fontsize=20,
            ha="center", va="center", alpha=0),
    ax.text(3.42, 2408, "CH$_4$", color="#ff5b57", fontsize=20,
            ha="center", va="center", alpha=0),
    ax.text(4.02, 2432, "SO$_2$", color="#ff5b57", fontsize=20,
            ha="center", va="center", alpha=0),
    ax.text(4.43, 2230, "CO$_2$", color="#ffd166", fontsize=20,
            ha="center", va="center", alpha=0),
]

# ---------------------------------------------------
# Static residual confidence lines
# ---------------------------------------------------

axr.axhline(0, color=FG, lw=1.2, alpha=0.65)
axr.axhline(1, color=FG, lw=1.0, alpha=0.45)
axr.axhline(-1, color=FG, lw=1.0, alpha=0.45)
axr.axhline(2, color=FG, lw=1.0, ls="--", alpha=0.55)
axr.axhline(-2, color=FG, lw=1.0, ls="--", alpha=0.55)

# ---------------------------------------------------
# Animated model lines
# ---------------------------------------------------

line_best, = ax.plot([], [], color=FG, lw=2.8, label="Best fit model", zorder=5)
line_water, = ax.plot([], [], color="#3b82ff", lw=1.7, ls="--", label="No water", zorder=4)
line_methane, = ax.plot([], [], color="#ff5b57", lw=1.7, ls="--", label="No methane", zorder=4)
line_co2, = ax.plot([], [], color="#ffd166", lw=1.7, ls="--", label="No CO2", zorder=4)
line_so2, = ax.plot([], [], color="#b33a3a", lw=1.7, ls="--", label="No SO2", zorder=4)

# ---------------------------------------------------
# Animated data layers
# ---------------------------------------------------

niriss_err_artist = ax.errorbar(
    [],
    [],
    yerr=[],
    fmt="o",
    ms=6,
    color="#ff3131",
    ecolor=FG,
    elinewidth=1.0,
    capsize=2,
    markeredgecolor=FG,
    markeredgewidth=0.5,
    label="NIRISS SOSS",
    zorder=8
)

nirspec_err_artist = ax.errorbar(
    [],
    [],
    yerr=[],
    fmt="o",
    ms=6,
    color="#ffae1a",
    ecolor=FG,
    elinewidth=1.0,
    capsize=2,
    markeredgecolor=FG,
    markeredgewidth=0.5,
    label="NIRSpec G395H",
    zorder=8
)

phot_err_artist = ax.errorbar(
    [],
    [],
    yerr=[],
    fmt="o",
    ms=7,
    color="#2f5bff",
    ecolor=FG,
    elinewidth=1.0,
    capsize=2,
    markeredgecolor=FG,
    markeredgewidth=0.5,
    label="CHEOPS + TESS",
    zorder=9
)

res_scatter = axr.scatter(
    [],
    [],
    s=20,
    color=FG,
    alpha=0.9,
    zorder=5
)

# ---------------------------------------------------
# Title / legend
# ---------------------------------------------------

title = ax.text(
    0.55,
    2450,
    "TOI-1130 b transmission spectrum",
    color=FG,
    fontsize=20,
    ha="left",
    va="top"
)

subtitle = ax.text(
    0.55,
    2430,
    "",
    color=FG,
    fontsize=13,
    alpha=0.75,
    ha="left",
    va="top"
)

legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.16),
    ncol=4,
    frameon=True,
    fontsize=12
)

legend.get_frame().set_facecolor("#111827")
legend.get_frame().set_edgecolor("#4b5563")
legend.get_frame().set_alpha(0.95)

for text in legend.get_texts():
    text.set_color(FG)

# ---------------------------------------------------
# Helpers for animated errorbars
# ---------------------------------------------------

def clear_errorbar(container):
    line, caplines, barlinecols = container
    line.set_data([], [])

    for cap in caplines:
        cap.set_data([], [])

    for barlinecol in barlinecols:
        barlinecol.set_segments([])

def set_errorbar(container, x, y, yerr):
    line, caplines, barlinecols = container

    line.set_data(x, y)

    segments = [
        [(xi, yi - ei), (xi, yi + ei)]
        for xi, yi, ei in zip(x, y, yerr)
    ]

    if barlinecols:
        barlinecols[0].set_segments(segments)

    cap_size = 0.018

    if len(caplines) >= 2:
        lower = [(xi - cap_size, yi - ei, xi + cap_size, yi - ei)
                 for xi, yi, ei in zip(x, y, yerr)]
        upper = [(xi - cap_size, yi + ei, xi + cap_size, yi + ei)
                 for xi, yi, ei in zip(x, y, yerr)]

        caplines[0].set_data(
            [p for seg in lower for p in (seg[0], seg[2], np.nan)],
            [p for seg in lower for p in (seg[1], seg[3], np.nan)]
        )

        caplines[1].set_data(
            [p for seg in upper for p in (seg[0], seg[2], np.nan)],
            [p for seg in upper for p in (seg[1], seg[3], np.nan)]
        )

# ---------------------------------------------------
# Animation
# ---------------------------------------------------

fps = 24
duration_seconds = 12
frames = fps * duration_seconds

def update(frame):
    p = frame / (frames - 1)

    # Phase 1: molecular bands appear
    band_alpha = np.clip((p - 0.03) / 0.18, 0, 1) * 0.16

    for patch, _, _, _, _ in band_patches:
        patch.set_alpha(band_alpha)

    for txt in mol_labels:
        txt.set_alpha(np.clip((p - 0.15) / 0.15, 0, 1))

    # Phase 2: model lines draw left-to-right
    model_progress = np.clip((p - 0.18) / 0.35, 0, 1)
    n_model = max(2, int(model_progress * len(w)))

    wx = w[:n_model]

    line_best.set_data(wx, model[:n_model])
    line_water.set_data(wx, model_no_water[:n_model])
    line_methane.set_data(wx, model_no_methane[:n_model])
    line_co2.set_data(wx, model_no_co2[:n_model])
    line_so2.set_data(wx, model_no_so2[:n_model])

    # Phase 3: observed points appear
    data_progress = np.clip((p - 0.45) / 0.45, 0, 1)
    current_w = 0.5 + data_progress * (5.2 - 0.5)

    mask_phot = phot_w <= current_w
    mask_niriss = niriss_w <= current_w
    mask_nirspec = nirspec_w <= current_w

    set_errorbar(
        phot_err_artist,
        phot_w[mask_phot],
        phot_y[mask_phot],
        phot_err[mask_phot]
    )

    set_errorbar(
        niriss_err_artist,
        niriss_w[mask_niriss],
        niriss_y[mask_niriss],
        niriss_err[mask_niriss]
    )

    set_errorbar(
        nirspec_err_artist,
        nirspec_w[mask_nirspec],
        nirspec_y[mask_nirspec],
        nirspec_err[mask_nirspec]
    )

    mask_res = all_w <= current_w
    res_scatter.set_offsets(
        np.column_stack([
            all_w[mask_res],
            resid[mask_res]
        ])
    )

    if p < 0.35:
        subtitle.set_text("Molecular absorption bands")
    elif p < 0.55:
        subtitle.set_text("Best-fit model and molecule-removed models")
    else:
        subtitle.set_text("JWST + CHEOPS + TESS transmission spectrum")

    return (
        line_best,
        line_water,
        line_methane,
        line_co2,
        line_so2,
        res_scatter,
        subtitle,
        *mol_labels
    )

anim = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=1000 / fps,
    blit=False
)

# ---------------------------------------------------
# Save GIF
# ---------------------------------------------------

writer = PillowWriter(fps=fps)

anim.save(
    OUTPUT_GIF,
    writer=writer,
    savefig_kwargs={
        "facecolor": BG,
        "pad_inches": 0.05
    }
)

print(f"Saved: {OUTPUT_GIF}")

plt.close(fig)